# Study and arm characteristics

Source notebook for table 1

PTCy meta-analysis collaboration

This notebook rebuilds table 1 of the article directly from the extraction database, so every count in the published table can be traced to the rows it came from.

Inputs, copied verbatim from `02_extraction/` of the analysis repository (<https://github.com/Russlewisbo/ptcy_metaanalys>):

| File               | Rows | Contents                         |
|:-------------------|-----:|:---------------------------------|
| `data/studies.csv` |  251 | one row per included publication |
| `data/arms.csv`    |  525 | one row per treatment arm        |
| `data/rob.csv`     |    — | ROBINS-I / RoB 2 judgements      |

In [ ]:
library(dplyr)



Attaching package: 'dplyr'

The following objects are masked from 'package:stats':

    filter, lag

The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union

studies: 251 (comparative 225, single-arm descriptive 26)
arms: 525 (PTCy 277, comparator 248)

## Study-level counts

In [ ]:

pct <- function(n, d) sprintf("%d (%d%%)", n, round(100 * n / d))

fu <- as_num(comparative$follow_up_median_months)
cat(sprintf("follow-up parsed for %d of %d comparative studies (%d not reported/unparseable)\n",
            sum(!is.na(fu)), length(fu), sum(is.na(fu))))


follow-up parsed for 146 of 225 comparative studies (79 not reported/unparseable)

  Characteristic                    Value
  --------------------------------- -------------------
  Studies included in the review    251
  Comparative (≥2 arms)             225
  Single-arm descriptive            26
  Total patients                    178 146
  Randomised controlled trials      14 (6%)
  Retrospective cohort              132 (59%)
  Registry analysis                 62 (28%)
  Prospective cohort                17 (8%)
  Publication year, median (IQR)    2023 (2021--2024)
  Follow-up, months, median (IQR)   26 (22--36)


## Arm-level characteristics

Continuous variables are summarised as the median of the arm-level values; the IQR therefore describes between-arm variability, not between-patient variability.

In [ ]:

med_iqr <- function(x, d = 1) {
  x <- as_num(x)
  x <- x[!is.na(x)]
  if (!length(x)) return("—")
  q <- quantile(x, c(.25, .5, .75))
  sprintf("%.*f (%.*f–%.*f)", d, q[2], d, q[1], d, q[3])
}

# a categorical "predominant" column summarised as n (%) within arm role
cat_rows <- function(col, labels) {
  arms |>
    mutate(lvl = .data[[col]]) |>
    count(arm_role, lvl) |>
    group_by(arm_role) |>
    mutate(txt = sprintf("%d (%d%%)", n, round(100 * n / sum(n)))) |>
    ungroup() |>
    select(arm_role, lvl, txt) |>
    pivot_wider(names_from = arm_role, values_from = txt) |>
    filter(lvl %in% names(labels)) |>
    mutate(lvl = unname(labels[lvl])) |>
    rename(Characteristic = lvl)
}

num_row <- function(label, col, d = 1) {
  s <- arms |> group_by(arm_role) |> summarise(v = med_iqr(.data[[col]], d), .groups = "drop")
  tibble(Characteristic = label,
         PTCy_arm       = s$v[s$arm_role == "PTCy_arm"],
         comparator_arm = s$v[s$arm_role == "comparator_arm"])
}

donor <- c(haplo = "  Haploidentical", mud = "  MUD (10/10)", mmud = "  MMUD (9/10 or lower)",
           msd = "  MSD", ucb = "  UCB")
graft <- c(pbsc = "  PBSC", bm = "  BM", ucb = "  UCB")
cond  <- c(mac = "  MAC", ric = "  RIC", nma = "  NMA")

bind_rows(
  num_row("Patients per arm, median (IQR)", "n_patients", 0),
  num_row("Age, years, median (IQR)",       "age_median_years"),
  num_row("Male, %",                        "pct_male"),
  tibble(Characteristic = "Donor type (predominant)", PTCy_arm = "", comparator_arm = ""),
  cat_rows("donor_predominant", donor),
  tibble(Characteristic = "Graft source (predominant)", PTCy_arm = "", comparator_arm = ""),
  cat_rows("graft_predominant", graft),
  tibble(Characteristic = "Conditioning (predominant)", PTCy_arm = "", comparator_arm = ""),
  cat_rows("conditioning_predominant", cond),
  num_row("CMV R+, %",           "cmv_recipient_pos_pct"),
  num_row("Steroid exposure, %", "systemic_steroid_for_gvhd_exposure_pct")
) |>
  replace_na(list(PTCy_arm = "—", comparator_arm = "—")) |>
  rename(`PTCy arms` = PTCy_arm, `Comparator arms` = comparator_arm) |>
  kable(align = "lrr")


  Characteristic                             PTCy arms     Comparator arms
  -------------------------------- ------------------- -------------------
  Patients per arm, median (IQR)          96 (47--209)       120 (51--375)
  Age, years, median (IQR)           52.0 (45.0--58.0)   53.0 (43.0--57.9)
  Male, %                            57.3 (54.0--61.7)   57.4 (53.0--61.8)
  Donor type (predominant)                             
  Haploidentical                             130 (47%)            36 (15%)
  Graft source (predominant)                           
  Conditioning (predominant)                           
  CMV R+, %                          74.6 (66.0--80.0)   69.4 (59.5--81.3)
  Steroid exposure, %                26.6 (20.0--32.7)   34.5 (28.5--43.5)


## Risk of bias

In [ ]:

rob_col <- intersect(c("overall_judgement", "overall", "rob_overall"), names(rob))[1]
if (!is.na(rob_col)) {
  rob |>
    count(Judgement = .data[[rob_col]]) |>
    mutate(`n (%)` = sprintf("%d (%d%%)", n, round(100 * n / sum(n)))) |>
    select(-n) |>
    kable()
} else {
  cat("Columns present in rob.csv:", paste(names(rob), collapse = ", "), "\n")
}


  Judgement       n (%)
  --------------- -----------
  high            2 (1%)
  low             1 (0%)
  moderate        71 (29%)
  serious         150 (62%)
  some_concerns   11 (5%)
  NA              6 (2%)
